# Number Systems — Exercises

Practice problems aligned with `theory.ipynb`. Each exercise has:
- 🎯 **Problem** with real ML context
- 📝 **Scaffold** (fill in `# YOUR CODE HERE`)
- ✅ **Solution** (in separate cell)

| Level | Description | Count |
|-------|-------------|-------|
| ⭐ | Essential — core understanding | 3 |
| ⭐⭐ | Applied — real ML engineering | 4 |
| ⭐⭐⭐ | Interview — production debugging | 3 |

In [ ]:
import numpy as np
import struct
import warnings
np.set_printoptions(precision=6, suppress=True)

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    print(f'NumPy {np.__version__} | PyTorch {torch.__version__} — Ready!')
except ImportError:
    HAS_TORCH = False
    print(f'NumPy {np.__version__} — Ready! (PyTorch exercises will be skipped)')

---

## Exercise 1: Float Comparison ⭐

You're building a training loop that stops when loss converges.

**Task**: Write `has_converged(current, previous, tol=1e-6)` that:
- Returns `True` if values are approximately equal
- Handles edge cases: `NaN`, `inf`, negative losses
- **Never uses `==` for float comparison**

In [ ]:
# Exercise 1: Your Solution

def has_converged(current, previous, tol=1e-6):
    """
    Check if loss has converged.
    Returns True if current ≈ previous within tolerance.
    Returns False if either value is NaN.
    """
    # YOUR CODE HERE
    pass

# Test cases
print(has_converged(0.001, 0.001))           # True
print(has_converged(0.001, 0.0010000001))    # True
print(has_converged(0.001, 0.002))           # False
print(has_converged(float('nan'), 0.001))    # False
print(has_converged(float('inf'), float('inf')))  # True

In [ ]:
# Exercise 1: Solution ✅

def has_converged(current, previous, tol=1e-6):
    """Check convergence with proper float comparison."""
    if np.isnan(current) or np.isnan(previous):
        return False
    if np.isinf(current) or np.isinf(previous):
        return current == previous  # inf == inf is True
    return bool(np.isclose(current, previous, rtol=tol, atol=tol))

# Verify all cases
assert has_converged(0.001, 0.001) == True
assert has_converged(0.001, 0.0010000001) == True
assert has_converged(0.001, 0.002) == False
assert has_converged(float('nan'), 0.001) == False
assert has_converged(float('inf'), float('inf')) == True
print('All tests passed ✓')
print('\n→ Key lesson: NEVER use == for floats in ML code')
print('→ np.isclose uses both relative and absolute tolerance')

---

## Exercise 2: Inspect Float Binary Layout ⭐

Understanding IEEE 754 helps debug precision bugs. Use `struct.pack` to see the actual bits.

**Task**: Write `inspect_float(value)` that prints:
- Sign bit (1 bit)
- Exponent (8 bits for float32)
- Mantissa (23 bits for float32)
- Whether the number is normal, subnormal, zero, inf, or NaN

In [ ]:
# Exercise 2: Your Solution

def inspect_float(value):
    """
    Inspect IEEE 754 float32 binary representation.
    Uses struct.pack('>f', value) to get raw bytes.
    """
    # YOUR CODE HERE
    # Hint:
    # 1. packed = struct.pack('>f', value) gives 4 bytes
    # 2. Convert bytes to bits
    # 3. Split: sign=bits[0], exponent=bits[1:9], mantissa=bits[9:32]
    pass

# Test
inspect_float(1.0)
inspect_float(-0.5)
inspect_float(0.1)       # interesting: repeating binary!
inspect_float(float('inf'))
inspect_float(float('nan'))

In [ ]:
# Exercise 2: Solution ✅

def inspect_float(value):
    """Inspect IEEE 754 float32 binary representation."""
    packed = struct.pack('>f', value)
    bits = ''.join(f'{byte:08b}' for byte in packed)
    
    sign = bits[0]
    exponent = bits[1:9]
    mantissa = bits[9:]
    exp_val = int(exponent, 2)
    
    # Classify
    if exp_val == 0 and int(mantissa, 2) == 0:
        kind = 'zero'
    elif exp_val == 0:
        kind = 'subnormal'
    elif exp_val == 255 and int(mantissa, 2) == 0:
        kind = 'infinity'
    elif exp_val == 255:
        kind = 'NaN'
    else:
        kind = f'normal (2^{exp_val - 127})'
    
    print(f'{value:>10} → S={sign} E={exponent} M={mantissa}  [{kind}]')

print(f'{"Value":>10}    Sign  Exponent  Mantissa (23 bits)')
print('-' * 65)
for v in [1.0, -0.5, 0.1, 3.14, float('inf'), float('nan'), 0.0]:
    inspect_float(v)

print('\n→ Notice: 0.1 has a non-trivial mantissa (repeating in binary)')
print('→ inf: all exponent bits = 1, mantissa = 0')
print('→ NaN: all exponent bits = 1, mantissa ≠ 0')

---

## Exercise 3: Numerically Stable Softmax ⭐⭐

Your neural network classifier produces logits of 1000+. The naive softmax overflows.

**Task**: Implement `stable_softmax(x)` that:
- Works with any input (including logits of 1000+)
- Returns valid probabilities (sum to 1, no NaN/inf)
- Supports batches (2D input, softmax along last axis)

$$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

In [ ]:
# Exercise 3: Your Solution

def stable_softmax(x):
    """Numerically stable softmax (1D or 2D batch)."""
    # YOUR CODE HERE
    # Hint: x - np.max(x, axis=-1, keepdims=True)
    pass

# Tests
x1 = np.array([1.0, 2.0, 3.0])
x2 = np.array([1000.0, 1001.0, 1002.0])
x3 = np.array([[1, 2, 3], [1000, 1001, 1002]])

print(f'Normal: {stable_softmax(x1)}  sum={stable_softmax(x1).sum():.4f}')
print(f'Large:  {stable_softmax(x2)}  sum={stable_softmax(x2).sum():.4f}')
print(f'Batch:\n{stable_softmax(x3)}')

In [ ]:
# Exercise 3: Solution ✅

def stable_softmax(x):
    """Numerically stable softmax."""
    x = np.asarray(x, dtype=np.float64)
    x_shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# Verify
x2 = np.array([1000.0, 1001.0, 1002.0])
result = stable_softmax(x2)
assert not np.any(np.isnan(result)), 'NaN detected!'
assert np.isclose(result.sum(), 1.0), 'Does not sum to 1!'
print(f'Stable softmax([1000,1001,1002]) = {result}')
print(f'Sum = {result.sum():.6f} ✓')

# Compare with PyTorch
if HAS_TORCH:
    pt = torch.nn.functional.softmax(torch.tensor(x2), dim=0).numpy()
    print(f'PyTorch F.softmax:                {pt}')
    print(f'Match: {np.allclose(result, pt)} ✓')

---

## Exercise 4: Log Probability (Underflow Prevention) ⭐⭐

A language model computes: P(sentence) = P(w₁) × P(w₂|w₁) × ... × P(wₙ|w₁..ₙ₋₁)

This underflows to 0 for long sequences.

**Task**: Implement `compare_sequences(probs_a, probs_b)` that:
- Computes log-probability for each sequence
- Returns which sequence is more likely
- Works even with 1000+ word sequences

In [ ]:
# Exercise 4: Your Solution

def sequence_log_prob(word_probs):
    """Log probability of a word sequence."""
    # YOUR CODE HERE
    pass

def compare_sequences(probs_a, probs_b):
    """Return 'A', 'B', or 'equal' based on which is more likely."""
    # YOUR CODE HERE
    pass

# Test: which sentence is more likely?
# Sentence A: 50 words, each with P=0.02
# Sentence B: 50 words, each with P=0.015
probs_a = [0.02] * 50
probs_b = [0.015] * 50

print(f'Direct product A: {np.prod(probs_a)}')  # underflows!
print(f'Direct product B: {np.prod(probs_b)}')  # underflows!
print(f'More likely: {compare_sequences(probs_a, probs_b)}')

In [ ]:
# Exercise 4: Solution ✅

def sequence_log_prob(word_probs):
    """Log probability: log(p1×p2×...) = log(p1)+log(p2)+..."""
    return np.sum(np.log(np.asarray(word_probs)))

def compare_sequences(probs_a, probs_b):
    lp_a = sequence_log_prob(probs_a)
    lp_b = sequence_log_prob(probs_b)
    print(f'  log P(A) = {lp_a:.2f}')
    print(f'  log P(B) = {lp_b:.2f}')
    if np.isclose(lp_a, lp_b): return 'equal'
    return 'A' if lp_a > lp_b else 'B'

# Verify
probs_a = [0.02] * 50
probs_b = [0.015] * 50
print(f'Winner: Sentence {compare_sequences(probs_a, probs_b)}')
print('\n→ Higher log-prob = more likely')
print('→ This is how beam search works in LLMs!')

---

## Exercise 5: Stable Cross-Entropy Loss ⭐⭐⭐

**Interview-level**: Implement numerically stable cross-entropy.

Naive: $L = -\sum_i y_i \log(\text{softmax}(x)_i)$ — overflows + `log(0)`!

**Task**: Implement `stable_cross_entropy(logits, labels)` using the identity:

$$\log(\text{softmax}(x)_i) = x_i - \log\sum_j e^{x_j}$$

This avoids computing softmax directly!

In [ ]:
# Exercise 5: Your Solution

def stable_cross_entropy(logits, labels):
    """
    Numerically stable cross-entropy loss.
    logits: raw model outputs (before softmax)
    labels: one-hot encoded targets
    """
    # YOUR CODE HERE
    # Hint: logsumexp(x) = max(x) + log(sum(exp(x - max(x))))
    pass

# Test with extreme logits
logits = np.array([1000.0, 1001.0, 1002.0])
labels = np.array([0, 0, 1])  # class 2 is correct
print(f'Loss: {stable_cross_entropy(logits, labels)}')

In [ ]:
# Exercise 5: Solution ✅

def stable_cross_entropy(logits, labels):
    """Stable CE using: log_softmax(x) = x - logsumexp(x)"""
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    # Stable logsumexp
    m = np.max(logits)
    lse = m + np.log(np.sum(np.exp(logits - m)))
    # log_softmax = logits - logsumexp
    log_probs = logits - lse
    return -np.sum(labels * log_probs)

# Verify with extreme values
logits = np.array([1000.0, 1001.0, 1002.0])
labels = np.array([0, 0, 1])
loss = stable_cross_entropy(logits, labels)
print(f'Loss: {loss:.4f}')
assert not np.isnan(loss), 'NaN detected!'

# Compare with PyTorch
if HAS_TORCH:
    pt_loss = nn.CrossEntropyLoss()(torch.tensor([logits]), torch.tensor([2])).item()
    print(f'PyTorch CE: {pt_loss:.4f}')
    print(f'Match: {np.isclose(loss, pt_loss, atol=0.01)} ✓')
print('\n→ Key: log_softmax = logits - logsumexp (never compute softmax directly)')

---

## Exercise 6: Model Memory Calculator ⭐

Before training, you need to know: **will my model fit in GPU memory?**

**Task**: Write `training_memory_gb(params, dtype, optimizer='adam')` that calculates:
- Model weights memory
- Gradient memory (same size as weights)
- Optimizer states (Adam: 2× extra for momentum + variance)
- **Total training memory**

| Component | Size |
|-----------|------|
| Weights | N × bytes_per_param |
| Gradients | N × bytes_per_param |
| Adam states | 2 × N × 4 bytes (always float32) |

In [ ]:
# Exercise 6: Your Solution

def training_memory_gb(num_params, dtype=np.float32, optimizer='adam'):
    """
    Calculate total training memory in GB.
    Include: weights + gradients + optimizer states.
    Adam keeps 2 extra float32 copies (momentum, variance).
    """
    # YOUR CODE HERE
    pass

# Test with real models
models = {
    'GPT-2': 117_000_000,
    'LLaMA-7B': 7_000_000_000,
    'LLaMA-70B': 70_000_000_000,
}
for name, params in models.items():
    mem = training_memory_gb(params)
    print(f'{name}: {params/1e9:.1f}B params → {mem:.1f} GB training memory')

In [ ]:
# Exercise 6: Solution ✅

def training_memory_gb(num_params, dtype=np.float32, optimizer='adam'):
    """Total training memory including optimizer states."""
    bytes_per_param = np.dtype(dtype).itemsize
    GB = 1024**3
    
    weights = num_params * bytes_per_param
    gradients = num_params * bytes_per_param
    # Adam stores momentum + variance in float32 regardless of model dtype
    opt_states = 2 * num_params * 4 if optimizer == 'adam' else 0
    
    total = weights + gradients + opt_states
    return total / GB

# Full comparison table
models = {'GPT-2': 117e6, 'LLaMA-7B': 7e9, 'LLaMA-70B': 70e9}
print(f'{"Model":<12} {"FP32 Train":>12} {"BF16 Train":>12} {"FP32 Infer":>12} {"INT4 Infer":>12}')
print('-'*52)
for name, p in models.items():
    t32 = training_memory_gb(p, np.float32)
    t16 = training_memory_gb(p, np.float16)
    i32 = p * 4 / 1024**3
    i4 = p * 0.5 / 1024**3
    print(f'{name:<12} {t32:>9.1f} GB  {t16:>9.1f} GB  {i32:>9.1f} GB  {i4:>9.1f} GB')

print('\n→ Training needs ~4× more memory than inference (gradients + optimizer)')
print('→ LLaMA-70B in FP32 needs ~1 TB — requires multi-GPU!')
print('→ INT4 inference: 70B model fits in ~35GB (single GPU)')

---

## Exercise 7: Per-Channel Quantization ⭐⭐

Per-tensor quantization loses accuracy when channels have different weight magnitudes.

**Task**: Implement `quantize_per_channel(W)` that:
- Computes a separate scale per output channel (row)
- Quantizes each channel independently to int8
- Compare MSE with per-tensor quantization

In [ ]:
# Exercise 7: Your Solution

def quantize_per_channel(W):
    """
    Per-channel int8 quantization.
    W: shape (out_channels, in_features)
    Returns: (int8_weights, scales) where scales shape = (out_channels,)
    """
    # YOUR CODE HERE
    pass

# Test
np.random.seed(42)
W = np.random.randn(4, 256).astype(np.float32)
W[0] *= 10    # large weights
W[1] *= 0.01  # tiny weights

q, scales = quantize_per_channel(W)
print(f'Scales: {scales.round(4)}')
print(f'Ch0 int8 range: [{q[0].min()}, {q[0].max()}]')
print(f'Ch1 int8 range: [{q[1].min()}, {q[1].max()}]  (should use full range!)')

In [ ]:
# Exercise 7: Solution ✅

def quantize_per_channel(W):
    n_ch = W.shape[0]
    scales = np.zeros(n_ch, dtype=np.float32)
    q = np.zeros_like(W, dtype=np.int8)
    for c in range(n_ch):
        scales[c] = np.max(np.abs(W[c])) / 127
        if scales[c] > 0:
            q[c] = np.clip(np.round(W[c] / scales[c]), -128, 127).astype(np.int8)
    return q, scales

# Compare per-tensor vs per-channel
np.random.seed(42)
W = np.random.randn(4, 256).astype(np.float32)
W[0] *= 10; W[1] *= 0.01

# Per-tensor
s_t = np.max(np.abs(W)) / 127
q_t = np.clip(np.round(W / s_t), -128, 127).astype(np.int8)
mse_t = np.mean((W - q_t * s_t)**2)

# Per-channel
q_c, s_c = quantize_per_channel(W)
mse_c = np.mean((W - q_c.astype(np.float32) * s_c[:, None])**2)

print(f'Per-tensor MSE:  {mse_t:.6f}')
print(f'Per-channel MSE: {mse_c:.6f}')
print(f'Improvement:     {mse_t/mse_c:.1f}×')
print('\n→ Ch1 (tiny weights) gets crushed by per-tensor\'s global scale')
print('→ Per-channel gives each channel its own scale → much better!')

---

## Exercise 8: Stochastic Rounding ⭐⭐

In FP8 training, small gradient updates always round to zero → weights never change.

**Task**: Implement `stochastic_round(x)` that:
- Rounds UP with probability = fractional part
- Rounds DOWN with probability = 1 - fractional part
- Is **unbiased**: E[round(x)] = x

Then demonstrate that it preserves gradient updates that deterministic rounding loses.

In [ ]:
# Exercise 8: Your Solution

def stochastic_round(x):
    """
    Stochastic rounding: floor(x) + Bernoulli(frac(x))
    """
    # YOUR CODE HERE
    pass

# Test: accumulate 100 gradient updates of 0.15
np.random.seed(42)
w_det, w_stoch = 0.0, 0.0

for _ in range(100):
    w_det = np.round(w_det + 0.15)
    w_stoch = float(stochastic_round(np.array([w_stoch + 0.15])))

print(f'Expected:       {0.15 * 100:.1f}')
print(f'Deterministic:  {w_det:.1f}')
print(f'Stochastic:     {w_stoch:.1f}')

In [ ]:
# Exercise 8: Solution ✅

def stochastic_round(x):
    """Unbiased rounding via random Bernoulli samples."""
    x = np.asarray(x, dtype=np.float64)
    floor = np.floor(x)
    frac = x - floor
    return floor + (np.random.random(x.shape) < frac).astype(x.dtype)

# 1. Verify unbiasedness
np.random.seed(42)
vals = np.full(10000, 3.7)
det = np.round(vals)
stoch = stochastic_round(vals)
print(f'Rounding 3.7 (10,000 times):')
print(f'  Deterministic: mean={det.mean():.1f} (biased: always 4)')
print(f'  Stochastic:    mean={stoch.mean():.3f} (unbiased: ~3.7) ✓')

# 2. Gradient accumulation test
np.random.seed(42)
w_det, w_stoch = 0.0, 0.0
for _ in range(100):
    w_det = np.round(w_det + 0.15)
    w_stoch = float(stochastic_round(np.array([w_stoch + 0.15])))

print(f'\n100 updates of +0.15:')
print(f'  Expected:      {15.0:.1f}')
print(f'  Deterministic: {w_det:.1f}  ← updates LOST!')
print(f'  Stochastic:    {w_stoch:.1f}  ← preserved! ✓')

---

## Exercise 9: Kahan Summation ⭐⭐⭐

**Interview-level**: You're accumulating gradients across 1M micro-batches in float32.

**Task**: Implement `kahan_sum(values)` that:
- Tracks rounding error with a compensation variable
- Achieves better accuracy than naive accumulation
- Works in original precision (no float64 upcast)

```
for each value:
    y = value - compensation
    temp = total + y
    compensation = (temp - total) - y  # what was lost
    total = temp
```

In [ ]:
# Exercise 9: Your Solution

def kahan_sum(values):
    """Kahan compensated summation."""
    # YOUR CODE HERE
    pass

# Test: sum 1 million tiny float32 values
vals = np.full(1_000_000, 1e-5, dtype=np.float32)
result = kahan_sum(vals)
print(f'Result: {result:.6f} (expected: 10.0)')

In [ ]:
# Exercise 9: Solution ✅

def kahan_sum(values):
    """Compensated summation — tracks and corrects rounding error."""
    total = 0.0
    comp = 0.0
    for v in values:
        y = float(v) - comp
        temp = total + y
        comp = (temp - total) - y  # lost low-order bits
        total = temp
    return total

# Compare methods
vals = np.full(1_000_000, 1e-5, dtype=np.float32)
true = 10.0

naive = np.float32(0.0)
for v in vals: naive += np.float32(v)

kahan = kahan_sum(vals)
numpy_sum = float(np.sum(vals))

print(f'{"Method":<18} {"Result":>12} {"Error":>12}')
print('-'*44)
print(f'{"True":<18} {true:>12.6f} {0:>12.6f}')
print(f'{"Naive float32":<18} {float(naive):>12.6f} {abs(float(naive)-true):>12.6f}')
print(f'{"Kahan":<18} {kahan:>12.6f} {abs(kahan-true):>12.6f}')
print(f'{"NumPy (pairwise)":<18} {numpy_sum:>12.6f} {abs(numpy_sum-true):>12.6f}')
print('\n→ Kahan: near-perfect accuracy by tracking rounding error')
print('→ Use for gradient accumulation in mixed-precision training')

---

## Exercise 10: Mixed Precision Debugging ⭐⭐⭐

**Interview-level**: Your training run produces NaN loss after switching to float16.

**Task**: Diagnose and fix the precision issue.
1. Identify which operation causes NaN in float16
2. Apply loss scaling to fix it
3. Show that bfloat16 avoids the problem entirely

In [ ]:
# Exercise 10: Your Solution

def debug_mixed_precision():
    """
    Simulate a training step in different precisions.
    Show where float16 breaks and how to fix it.
    """
    # Simulate: large logits → softmax → cross-entropy → gradient
    logits_f32 = np.float32(500.0)  # large logit
    
    # YOUR CODE HERE:
    # 1. Try np.exp(np.float16(500)) — what happens?
    # 2. Try loss scaling: multiply logit by scale, compute, divide back
    # 3. Show np.exp on bfloat16 equivalent range
    pass

debug_mixed_precision()

In [ ]:
# Exercise 10: Solution ✅

print('--- Float16 Precision Debugging ---\n')

# Problem 1: Float16 overflow
print('1. Overflow in float16:')
print(f'   float16 max = {np.finfo(np.float16).max}')  # 65504
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(f'   exp(float16(12)) = {np.exp(np.float16(12))}')  # OK
    print(f'   exp(float16(20)) = {np.exp(np.float16(20))}')  # inf!
print(f'   exp(float32(20)) = {np.exp(np.float32(20)):.0f}')  # fine

# Problem 2: Small gradients disappear
print('\n2. Small gradients in float16:')
for grad in [1e-4, 1e-5, 1e-6, 1e-7]:
    f16 = np.float16(grad)
    lost = ' ← LOST!' if f16 == 0 else ''
    print(f'   grad={grad:.0e} → float16={f16}{lost}')

# Fix: Loss scaling
print('\n3. Fix with loss scaling:')
grad = 1e-6
scale = 1024
print(f'   grad={grad:.0e} → float16={np.float16(grad)} (lost!)')
print(f'   scaled={grad*scale:.0e} → float16={np.float16(grad*scale)} (preserved!)')
print(f'   unscaled={np.float16(grad*scale)/scale} ✓')

# Why bfloat16 is better
print('\n4. Why BFloat16 is preferred:')
print(f'   float16  range: max={np.finfo(np.float16).max}')
print(f'   float32  range: max={np.finfo(np.float32).max}')
print('   bfloat16 range: max=3.39e+38 (same as float32!)')
print('   → BFloat16 has same range as FP32, so no overflow on large values')

if HAS_TORCH:
    print('\n5. PyTorch verification:')
    x = torch.tensor(500.0)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        print(f'   torch.exp(float16(500)) = {torch.exp(x.half())}')
    print(f'   torch.exp(bfloat16(500)) = {torch.exp(x.bfloat16())}')
    print('   → BFloat16 handles large values that break float16')

---

## Summary: Skills Practiced

| # | Exercise | Skill | Level |
|---|----------|-------|-------|
| 1 | Float Comparison | `np.isclose()`, NaN handling | ⭐ |
| 2 | Binary Inspection | `struct.pack`, IEEE 754 layout | ⭐ |
| 3 | Stable Softmax | Subtract-max trick | ⭐⭐ |
| 4 | Log Probabilities | Underflow prevention | ⭐⭐ |
| 5 | Cross-Entropy | Logsumexp, log-softmax | ⭐⭐⭐ |
| 6 | Memory Calculator | Training vs inference sizing | ⭐ |
| 7 | Per-Channel Quantization | Production deployment | ⭐⭐ |
| 8 | Stochastic Rounding | FP8 / low-precision training | ⭐⭐ |
| 9 | Kahan Summation | Gradient accumulation | ⭐⭐⭐ |
| 10 | Mixed Precision Debug | Float16 vs BFloat16, loss scaling | ⭐⭐⭐ |

---

### Next Steps
- **notes.md**: Detailed theory and mathematical background
- Continue to: **02-Sets-and-Logic**